# 04｜正样本、负样本与目标分配

前面已经知道，检测模型会产生许多候选预测，每条预测包含类别和边界框。但一张图片中的真实物体通常远少于候选预测。

因此，在计算检测损失之前，训练系统必须先回答：

> 每个预测应该向哪个真实物体学习，还是应该学习背景？

这个过程叫作目标分配。本课先学习所有检测器共有的逻辑，再比较传统密集检测与 DETR 的两种分配思想。

## 1. 目标分配来自一个数量不相等的问题

设一张图片中有 $M$ 个真实物体，模型产生 $N$ 条候选预测。通常：

$$
N\gg M
$$

例如：

$$
M=2,\qquad N=1000
$$

模型产生 1000 条预测，不代表图片里有 1000 个物体。大量预测只是模型检查不同位置、尺度或查询结果的候选答案。

真实标注不会自动告诉第 37 条预测应该负责哪一个物体，所以需要额外建立预测与真实目标之间的对应关系。

## 2. 目标分配究竟输出什么

目标分配的输入是：

- 模型产生的候选预测或预定义候选位置。
- 这张图片的真实类别和真实边界框。
- 当前检测器规定的匹配规则。

目标分配的输出是一张训练责任表：

$$
\text{候选预测}\rightarrow
\begin{cases}
\text{负责某个真实物体}\\
\text{负责背景}\\
\text{暂时忽略}
\end{cases}
$$

有了这张责任表，损失函数才知道每个预测的正确类别是什么、是否需要计算边界框损失，以及应该与哪个真实框比较。

## 3. 正样本、负样本和忽略样本

**本课继续统一使用“预测槽位”。其他资料中常见的“预测单位”是更宽泛的叫法，在当前语境下同样表示一条能够被独立分配监督的候选输出。**

### 正样本

被分配去负责某个真实物体的预测槽位称为正样本。它会获得真实物体的类别标签与边界框，并参与分类和边界框回归监督。

### 负样本

被分配为背景的预测槽位称为负样本。它通常获得背景、无目标或低目标性的分类监督，但没有真实边界框可供回归。

### 忽略样本

有些预测既不明确属于正样本，也不适合当成可靠背景。训练时可以暂时不让它贡献相关损失，以免给模型传递模糊监督。

## 4. 正样本不等于模型已经预测正确

这是非常重要的区别。

正样本的含义是：

> 训练系统指定这条预测负责学习某个真实目标。

它并不表示当前类别已经正确，也不表示当前边界框已经准确。恰恰因为它还可能预测得很差，才需要分类损失和框损失去纠正。

同样，负样本也不是指一张没有物体的图片。目标检测中的正负样本通常指图片内部的候选预测槽位，而不是整张训练图片。

## 5. 建立一个具体场景

假设一张图片中有两个真实目标：

| 真实目标 | 类别 | 真实框 |
|---|---|---|
| $G_1$ | 狗 | $b_1$ |
| $G_2$ | 汽车 | $b_2$ |

模型或检测结构提供了 6 个候选预测槽位：

$$
P_1,P_2,P_3,P_4,P_5,P_6
$$

当前先不关心这些候选来自网格、Anchor 还是 Object Query，只观察它们与两个真实框的空间重叠关系。

## 6. 计算候选框与真实框的 IoU

假设得到下面的 IoU 对照表：

| 候选预测 | 与狗框 $G_1$ 的 IoU | 与汽车框 $G_2$ 的 IoU |
|---:|---:|---:|
| $P_1$ | 0.82 | 0.02 |
| $P_2$ | 0.68 | 0.01 |
| $P_3$ | 0.12 | 0.76 |
| $P_4$ | 0.05 | 0.58 |
| $P_5$ | 0.44 | 0.08 |
| $P_6$ | 0.03 | 0.09 |

每一行回答：这个候选框分别与两个真实目标重叠得怎么样。例如 $P_1$ 与狗的 IoU 为 0.82，与汽车只有 0.02，所以从空间关系看，它更适合负责狗。

## 7. 使用一个教学用的简单分配规则

为了理解基本过程，暂时规定：

$$
\begin{cases}
\max_j\operatorname{IoU}(P_i,G_j)\geq0.5,
&P_i\text{ 为正样本，并负责 IoU 最大的真实目标}\\
\max_j\operatorname{IoU}(P_i,G_j)<0.4,
&P_i\text{ 为负样本}\\
0.4\leq\max_j\operatorname{IoU}(P_i,G_j)<0.5,
&P_i\text{ 为忽略样本}
\end{cases}
$$

这只是为了学习而设定的简单规则，不是所有 YOLO、Faster R-CNN 或其他检测器的统一规则。真实模型可能结合中心位置、尺度、类别分数或动态代价进行分配。

## 8. 按规则逐个判断

- $P_1$ 的最大 IoU 为 0.82，对应狗，所以它是负责狗的正样本。
- $P_2$ 的最大 IoU 为 0.68，也对应狗，所以它同样是负责狗的正样本。
- $P_3$ 和 $P_4$ 与汽车的最大 IoU 分别为 0.76 和 0.58，因此都是负责汽车的正样本。
- $P_5$ 的最大 IoU 为 0.44，落在模糊区间，因此暂时忽略。
- $P_6$ 与所有真实框的 IoU 都很低，因此作为背景负样本。

注意，这里允许同一个真实物体同时分配给多个正样本。

## 9. 得到最终责任表

| 候选预测 | 分配结果 | 分类目标 | 框回归目标 | 训练身份 |
|---:|---|---|---|---|
| $P_1$ | 负责 $G_1$ | 狗 | $b_1$ | 正样本 |
| $P_2$ | 负责 $G_1$ | 狗 | $b_1$ | 正样本 |
| $P_3$ | 负责 $G_2$ | 汽车 | $b_2$ | 正样本 |
| $P_4$ | 负责 $G_2$ | 汽车 | $b_2$ | 正样本 |
| $P_5$ | 暂不参与 | 无 | 无 | 忽略样本 |
| $P_6$ | 负责背景 | 背景或无目标 | 无 | 负样本 |

目标分配真正完成的工作，就是把原来的候选预测列表转换成这张可训练的监督表。

## 10. 分配完成后才能计算损失

对于正样本 $P_1$ 到 $P_4$，分类头学习对应的物体类别，边界框头学习对应的真实框。

对于负样本 $P_6$，分类或目标性分支学习背景、无目标，但不计算真实物体的边界框回归损失。

对于忽略样本 $P_5$，当前规则下不让它贡献相关训练损失。

所以顺序始终是：

$$
\text{候选预测与真实标注}
\rightarrow\text{目标分配}
\rightarrow\text{监督标签}
\rightarrow\text{检测损失}
$$

## 11. 为什么要保留负样本

如果训练时只使用正样本，模型只会学到“看到这些特征时应该输出物体”，却没有人教它普通背景不应该产生检测结果。

负样本让模型学习：墙面、天空、道路等背景不应该被当成物体，没有负责真实目标的候选位置应该降低目标性或输出背景。

但是密集检测器中的负样本通常远多于正样本。若所有负样本贡献都过强，训练可能被大量容易背景主导。因此许多检测器会使用采样、损失权重或 Focal Loss 等方法缓解类别不平衡。

当前只需要知道这个不平衡从哪里来，不展开解决公式。

## 12. 为什么有时需要忽略样本

以 $P_5$ 为例，它与狗框的 IoU 为 0.44：

- 说它足够好，可能有些勉强。
- 说它完全是背景，也不合理，因为它确实覆盖了部分狗。

如果强行把这种模糊候选当成负样本，就会要求模型压低一个靠近真实物体的预测；如果强行当成正样本，又可能给模型过于宽松的定位标准。

忽略区间的作用是不给边界附近的模糊候选强行贴标签。不同检测器是否使用忽略样本，以及怎样定义它，都会有所不同。

## 13. 什么叫一对多分配

观察责任表：

$$
\begin{aligned}
G_1&\leftarrow P_1,P_2\\
G_2&\leftarrow P_3,P_4
\end{aligned}
$$

一个真实物体可以分配给多个正样本，这叫作一对多分配。

这样做的好处是，一个真实物体可以从多个位置或尺度提供训练信号，密集检测器容易获得较充分的正样本监督。

但它也带来一个自然结果：推理时，多个位置可能同时对同一个物体给出高分预测，于是产生重复检测框。

## 14. 重复预测不是偶然故障

在上面的训练规则中，$P_1$ 和 $P_2$ 都被明确教导去预测同一只狗。

如果训练成功，它们都可能输出很高的狗类别分数，以及与真实狗框高度重叠的边界框。

因此，推理时出现多个相近的狗框，不一定是模型完全学坏了，而是一对多训练方式自然产生的结果。

传统检测流程通常在模型输出后再使用 NMS，从这些重复预测中保留一个代表结果。下一课会专门解释这一过程。

## 15. 什么叫一对一分配

一对一分配要求每个真实物体只分配给一个预测，每个预测最多负责一个真实物体。

对于上面的两个真实目标，理想对应可能是：

$$
\begin{aligned}
G_1&\leftrightarrow P_1\\
G_2&\leftrightarrow P_3
\end{aligned}
$$

其他预测都学习背景或无目标。

这种分配直接要求模型为每个真实物体留下一个最终代表，因此更符合“输出一个无重复目标集合”的思想。但怎样在全局范围选择最佳组合，会比简单逐个使用 IoU 阈值更复杂。

## 16. YOLO 类密集检测器的基本方向

YOLO 类模型通常在一个或多个特征图上产生大量密集预测。候选预测与图像空间位置、尺度或检测头设计有关。

训练时，目标分配器会依据当前版本的规则，为真实物体选择一个或多个正样本位置。历史版本可能较依赖网格和 Anchor，现代版本也可能采用无 Anchor 与动态分配方法。

因此，“YOLO 一定使用某个固定 IoU 阈值和固定 Anchor 规则”并不准确。我们当前需要理解的是共同方向：

$$
\text{大量密集候选}
\rightarrow\text{选择正负样本}
\rightarrow\text{计算分类与框损失}
$$

一对多监督和密集输出，使重复预测及后续筛选成为需要处理的问题。

## 17. DETR 的基本方向

DETR 使用固定数量的 Object Queries 产生预测，并把真实目标看成无序集合。

训练时，它不是分别检查每个 Query 是否超过某个简单 IoU 阈值，而是在整组 Query 预测与整组真实目标之间寻找总代价较小的一对一组合。

匹配代价会综合类别和边界框信息。匹配完成后：

- 被匹配的 Query 是正样本，学习对应类别和真实框。
- 未匹配的 Query 学习 no object。

具体的匈牙利匹配以后回到 DETR 时再推导。现在只需看清它与一对多分配的方向差异。

## 18. 目标分配与 NMS 不是同一件事

| 对比角度 | 目标分配 | NMS |
|---|---|---|
| 主要阶段 | 训练阶段 | 推理后处理阶段 |
| 是否需要真实标注 | 需要 | 不需要 |
| 主要输入 | 候选预测与真实目标 | 模型预测框与置信分数 |
| 主要作用 | 决定谁向谁学习 | 删除同一物体的重复预测 |

目标分配解释训练监督如何建立；NMS 解释模型预测完成后怎样整理重复结果。

它们有关联，因为一对多分配容易产生重复预测，但两者不能混为同一个步骤。

## 19. 目标分配只在训练时发生

训练时有真实类别与真实边界框，所以可以进行目标分配并计算损失。

推理新图片时没有真实答案，因此不存在“把预测分配给真实目标”这一步。模型只能直接输出自己认为可能的类别、框和分数，再按照检测器的推理规则进行筛选。

$$
\begin{aligned}
\text{训练}&:\text{预测}+\text{真实标注}\rightarrow\text{目标分配}\rightarrow\text{损失}\\
\text{推理}&:\text{预测}\rightarrow\text{置信度筛选与必要后处理}
\end{aligned}
$$

## 20. 常见误区

1. 正样本表示被指定去学习某个真实目标，不表示当前预测已经正确。
2. 检测中的负样本通常是图片内部被分配为背景的候选预测槽位，不一定是没有物体的整张图片。
3. IoU 可以提供空间依据，但目标分配还需要阈值、冲突处理或其他匹配代价。
4. 负样本主动学习背景，忽略样本通常不贡献相关损失。
5. YOLO 与 DETR 不只是损失公式不同，候选怎样产生、目标怎样分配也不同。

## 21. 本节小结

这一课需要真正记住七个结论：

1. 目标分配在候选预测与真实目标之间建立训练对应关系。
2. 正样本负责学习某个真实物体，但不代表当前预测已经正确。
3. 负样本学习背景或无目标，通常不计算真实框回归损失。
4. 忽略样本位于模糊区域，通常不贡献相关损失。
5. 一对多分配允许一个真实物体对应多个正样本，因此可能自然产生重复预测。
6. 一对一分配要求每个真实物体只对应一个预测，是 DETR 集合预测的关键基础。
7. 目标分配发生在训练时，NMS 则是传统检测器常见的推理后处理。

完整训练关系是：

$$
\text{候选预测}
\rightarrow\text{目标分配}
\rightarrow\text{正样本、负样本、忽略样本}
\rightarrow\text{分类与框监督}
\rightarrow\text{检测损失}
$$

下一课将专门学习：为什么一对多预测会留下许多重复框，以及 NMS 怎样一步步保留代表框。

## 22. 自测问题

1. 为什么检测损失前必须先做目标分配？
2. 正样本真正表示什么？它是否代表当前预测已经正确？
3. 负样本通常获得什么监督？
4. 为什么负样本通常没有真实框回归目标？
5. 忽略样本与负样本有什么区别？
6. 在教学例子中，$P_5$ 为什么被忽略？
7. 什么是一对多分配？为什么它容易产生重复预测？
8. 什么是一对一分配？
9. YOLO 类密集检测器与 DETR 的候选来源有什么不同？
10. DETR 中未匹配的 Query 学习什么？
11. 目标分配与 NMS 分别发生在哪个阶段？
12. 推理时为什么不能进行基于真实标注的目标分配？

### 自测参考答案

1. 不先建立对应关系，就不知道每条预测的类别标签，也不知道预测框该与哪个真实框比较。
2. 它表示当前预测槽位被指定去负责某个真实物体，不代表当前类别和框已经正确。
3. 背景、无目标或低目标性的分类监督。
4. 背景没有与之对应的真实物体边界框。
5. 负样本主动学习背景；忽略样本通常不贡献相关损失。
6. 它的最大 IoU 为 0.44，位于教学规则设置的模糊区间。
7. 一个真实物体可同时分配给多个正样本；多个预测因此都可能学会输出同一物体。
8. 每个真实物体只与一个预测对应，每个预测最多负责一个真实物体。
9. YOLO 类模型通常从特征图产生大量密集预测；DETR 从固定数量的 Object Queries 产生预测。
10. 学习 no object。
11. 目标分配发生在训练阶段；NMS 通常发生在推理后处理阶段。
12. 新图片没有真实类别和真实框，无法建立预测与真实目标的训练对应关系。